# LeetCode 3681 — Maximum XOR of Subsequences bằng Python

- Link: https://leetcode.com/problems/maximum-xor-of-subsequences/
- Độ khó: **Hard**
- Chủ đề: Bitmask, **cơ sở tuyến tính XOR** (linear basis / Gaussian elimination trên GF(2)), tham lam theo bit cao

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxXorSubsequences(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Rút gọn đề: `X ⊕ Y` với hai dãy con bất kỳ (được chồng nhau) = XOR của **một tập con bất kỳ** ⇒ bài **XOR tập con lớn nhất**.
2. Hiểu tập các giá trị XOR tập con là một **không gian vectơ** trên GF(2) (mỗi số là vectơ 30 bit).
3. Xây **cơ sở tuyến tính** (tối đa 30 vectơ) bằng cách chèn từng số, khử theo bit cao nhất.
4. Tham lam từ bit cao xuống để tìm giá trị lớn nhất trong không gian.
5. Kiểm chứng bằng vét cạn và bằng tập giá trị đạt được.

## 2. Đề bài & ví dụ

Chọn hai dãy con (có thể rỗng, có thể chồng nhau). `X`, `Y` là XOR của chúng. Tìm `max(X ⊕ Y)`.

**Ví dụ 1:** `[1,2,3]` → `3`.

**Ví dụ 2:** `[5,2]` → `7`.

**Ràng buộc:** `n ≤ 10^5`, `0 ≤ nums[i] ≤ 10^9` (`< 2^30`).

In [1]:
from typing import List
from functools import reduce
import random, time

examples = [([1, 2, 3], 3), ([5, 2], 7)]
for a, exp in examples:
    print(a, "-> kỳ vọng", exp)

[1, 2, 3] -> kỳ vọng 3
[5, 2] -> kỳ vọng 7


## 3. Kiến thức nền

### 3.1. Hai dãy con ⇒ một tập con

Gọi `A`, `B` là tập chỉ số của hai dãy con. Phần tử ở `A ∩ B` xuất hiện hai lần ⇒ triệt tiêu: `X ⊕ Y = XOR(A Δ B)` (hiệu đối xứng). Ngược lại, mọi tập `S` đạt được (lấy `A = S`, `B = ∅`). Thứ tự trong dãy con không quan trọng với XOR. Vậy đáp án = `max_{S} XOR(S)`.

In [2]:
nums = [1, 2, 3]
A = {1}; B = {1, 2}
X = reduce(lambda u, v: u ^ v, (nums[i] for i in A), 0); Y = reduce(lambda u, v: u ^ v, (nums[i] for i in B), 0)
print("X ^ Y =", X ^ Y, "| XOR(A Δ B) =", reduce(lambda u, v: u ^ v, (nums[i] for i in A ^ B), 0))

X ^ Y = 3 | XOR(A Δ B) = 3


### 3.2. Không gian XOR và cơ sở

Tập `{XOR(S)}` đóng với phép XOR ⇒ là không gian con của `GF(2)^30`. Cơ sở **dạng bậc thang**: `basis[b]` là vectơ có bit cao nhất là `b` (hoặc 0 nếu không có). Chèn `x`: duyệt bit từ cao xuống; nếu bit `b` của `x` bật và `basis[b] ≠ 0` thì `x ^= basis[b]`, ngược lại đặt `basis[b] = x` và dừng.

In [3]:
def build_basis(nums, B=30):
    basis = [0] * B
    for x in nums:
        for b in range(B - 1, -1, -1):
            if not x >> b & 1: continue
            if basis[b]: x ^= basis[b]
            else: basis[b] = x; break
    return basis

print([ (b, v) for b, v in enumerate(build_basis([5, 2, 7, 3])) if v ])

[(0, 1), (1, 2), (2, 5)]


### 3.3. Lấy giá trị lớn nhất

`res = 0`; duyệt `b` từ cao xuống: `res = max(res, res ^ basis[b])`. Vì `basis[b]` có bit cao nhất `b`, quyết định XOR hay không chỉ phụ thuộc bit `b` của `res` hiện tại (các bit cao hơn không đổi) ⇒ tham lam đúng.

### 3.4. Phụ thuộc tuyến tính trên GF(2)

Ba số `a, b, c` **phụ thuộc** nếu một tập con khác rỗng của chúng có XOR bằng 0 (ví dụ `1 ⊕ 2 ⊕ 3 = 0`). Thêm một số phụ thuộc vào tập **không** mở rộng không gian XOR. Khi chèn vào cơ sở, số phụ thuộc bị khử thành 0 và bị bỏ — đó là lý do cơ sở chỉ có tối đa 30 vectơ dù `n = 10^5`.

In [4]:
for trio in ([1, 2, 3], [1, 2, 4], [5, 6, 3]):
    print(trio, "phụ thuộc" if any(v == 0 for v in [trio[0] ^ trio[1] ^ trio[2]]) else "độc lập (bộ ba)",
          "| rank =", sum(1 for v in build_basis(trio) if v))

[1, 2, 3] phụ thuộc | rank = 2
[1, 2, 4] độc lập (bộ ba) | rank = 3
[5, 6, 3] phụ thuộc | rank = 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đáp án là phần tử lớn nhất của không gian sinh bởi `nums`. Không gian có tối đa `2^30` phần tử nhưng mô tả gọn bằng ≤ 30 vectơ cơ sở.

Ví dụ 2: `5 = 101₂`, `2 = 010₂` độc lập ⇒ lấy cả hai: `111₂ = 7`. ✔ Ví dụ 1: `3 = 1 ⊕ 2` phụ thuộc; không gian `{0,1,2,3}` ⇒ `3`. ✔

## 5. Cách 1 — Brute Force: mọi cặp dãy con

In [5]:
def brute(nums):
    n = len(nums); xs = []
    for mask in range(1 << n):
        x = 0
        for i in range(n):
            if mask >> i & 1: x ^= nums[i]
        xs.append(x)
    return max(a ^ b for a in xs for b in xs)

for a, exp in examples:
    print(brute(a), "kỳ vọng", exp)

3 kỳ vọng 3
7 kỳ vọng 7


**Nút thắt:** `4^n` cặp — chỉ `n ≤ 8`.

## 6. Ý tưởng thuật toán / bước đột phá

1. Quy về XOR tập con lớn nhất (3.1).
2. Xây cơ sở `O(n · 30)`.
3. Tham lam `O(30)`.

Tổng `O(30 n)` — với `n = 10^5` khoảng 3 triệu phép, chạy dưới nửa giây trong Python.

## 7. Các cách tiếp theo

- **Cách 2:** tập giá trị đạt được (DP trên tập) — `O(n · |span|)`, chỉ khi giá trị nhỏ.
- **Cách 3:** cơ sở bậc thang + tham lam (lời giải nộp).
- **Cách 4:** **Gauss–Jordan** (cơ sở rút gọn hoàn toàn): sau khi rút gọn, đáp án = XOR của **mọi** vectơ cơ sở.

### 7.1. Cách 2 — Tập giá trị đạt được

In [6]:
def span_set(nums):
    s = {0}
    for x in nums:
        s |= {v ^ x for v in s}
    return max(s)

for a, exp in examples:
    print(span_set(a), "kỳ vọng", exp)
sp = {0}
for x in [5, 2, 7, 3]: sp |= {v ^ x for v in sp}
print("không gian sinh bởi [5,2,7,3]:", sorted(sp), "| kích thước", len(sp), "= 2^rank với rank =", len(sp).bit_length() - 1)

3 kỳ vọng 3
7 kỳ vọng 7
không gian sinh bởi [5,2,7,3]: [0, 1, 2, 3, 4, 5, 6, 7] | kích thước 8 = 2^rank với rank = 3


**Nhận xét:** kích thước tập = `2^rank` ≤ `2^30` — với giá trị lớn và nhiều số độc lập, cách này bùng nổ.

### 7.2. Cách 3 — Cơ sở + tham lam (lời giải chính)

In [7]:
def basis_greedy(nums):
    basis = build_basis(nums)
    res = 0
    for b in range(29, -1, -1):
        if (res ^ basis[b]) > res: res ^= basis[b]
    return res

for a, exp in examples:
    print(basis_greedy(a), "kỳ vọng", exp)

3 kỳ vọng 3
7 kỳ vọng 7


### 7.3. Cách 4 — Gauss–Jordan

In [8]:
def gauss_jordan(nums):
    basis = build_basis(nums)
    for b in range(30):                                   # khử bit b khỏi mọi vectơ có bit cao hơn
        if not basis[b]: continue
        for c in range(b + 1, 30):
            if basis[c] >> b & 1: basis[c] ^= basis[b]
    return reduce(lambda u, v: u ^ v, basis, 0)

for a, exp in examples:
    print(gauss_jordan(a), "kỳ vọng", exp)

3 kỳ vọng 3
7 kỳ vọng 7


**Vì sao XOR tất cả?** Sau rút gọn, mỗi bit "trụ" `b` chỉ xuất hiện trong đúng `basis[b]`. XOR mọi vectơ bật mọi bit trụ — các bit trụ là các bit **cao nhất có thể quyết định**, nên đây là giá trị lớn nhất.

## 8. TRACE từng bước

In [9]:
def trace(nums, B=4):
    basis = [0] * B
    for x in nums:
        orig = x; steps = []
        for b in range(B - 1, -1, -1):
            if not x >> b & 1: continue
            if basis[b]: steps.append(f"^basis[{b}]={basis[b]:0{B}b}"); x ^= basis[b]
            else: basis[b] = x; steps.append(f"đặt basis[{b}]={x:0{B}b}"); break
        else:
            steps.append("-> 0 (phụ thuộc tuyến tính)")
        print(f"chèn {orig:0{B}b}: " + ", ".join(steps))
    res = 0
    for b in range(B - 1, -1, -1):
        if (res ^ basis[b]) > res: res ^= basis[b]; print(f"  bit {b}: lấy basis[{b}] -> res = {res:0{B}b}")
    print("đáp án:", res)

trace([5, 2, 7, 3])
trace([12, 10, 6])

chèn 0101: đặt basis[2]=0101
chèn 0010: đặt basis[1]=0010
chèn 0111: ^basis[2]=0101, ^basis[1]=0010, -> 0 (phụ thuộc tuyến tính)
chèn 0011: ^basis[1]=0010, đặt basis[0]=0001
  bit 2: lấy basis[2] -> res = 0101
  bit 1: lấy basis[1] -> res = 0111
đáp án: 7
chèn 1100: đặt basis[3]=1100
chèn 1010: ^basis[3]=1100, đặt basis[2]=0110
chèn 0110: ^basis[2]=0110, -> 0 (phụ thuộc tuyến tính)
  bit 3: lấy basis[3] -> res = 1100
đáp án: 12


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def maxXorSubsequences(self, nums: List[int]) -> int:
        basis = [0] * 30
        for x in nums:
            for b in range(29, -1, -1):
                if not (x >> b) & 1:
                    continue
                if basis[b]:
                    x ^= basis[b]
                else:
                    basis[b] = x
                    break
        res = 0
        for b in range(29, -1, -1):
            if res ^ basis[b] > res:
                res ^= basis[b]
        return res

sol = Solution()
for a, exp in examples:
    got = sol.maxXorSubsequences(a)
    print(a, "->", got, "✔" if got == exp else "✘")

[1, 2, 3] -> 3 ✔
[5, 2] -> 7 ✔


## 10. Vì sao thuật toán đúng

1. **Quy về tập con:** mục 3.1 — tập giá trị `X ⊕ Y` đúng bằng `{XOR(S) : S ⊆ chỉ số}`.
2. **Cơ sở sinh cùng không gian:** mỗi bước chèn thay `x` bằng `x ⊕ (tổ hợp các vectơ cơ sở)` — không đổi không gian sinh; nếu kết quả khác 0 thì thêm vào với bit cao nhất mới (độc lập tuyến tính).
3. **Tham lam:** giả sử đã xử lý các bit `> b` và `res` là giá trị lớn nhất có thể trên các bit đó trong không gian con sinh bởi `basis[>b]`. Vectơ duy nhất ảnh hưởng tới bit `b` mà không làm đổi bit cao hơn là `basis[b]` (và tổ hợp của nó với bit thấp hơn). Bật bit `b` nếu có thể luôn tốt hơn mọi lựa chọn bit thấp ⇒ chọn `res ^ basis[b]` khi lớn hơn.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(4^n)` | `O(2^n)` |
| 2. Tập đạt được | `O(n · 2^rank)` | `O(2^rank)` |
| 3. Cơ sở + tham lam | `O(30 n)` | `O(30)` |
| 4. Gauss–Jordan | `O(30 n + 30²)` | `O(30)` |

## 12. Bộ test

In [11]:
METHODS = [brute, span_set, basis_greedy, gauss_jordan, lambda a: Solution().maxXorSubsequences(a)]
for a, exp in examples:
    assert all(f(a) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [12]:
# 12.2 ca biên
assert Solution().maxXorSubsequences([0, 0]) == 0
assert Solution().maxXorSubsequences([7, 7]) == 7
assert Solution().maxXorSubsequences([2**29, 2**29 - 1]) == 2**30 - 1
print("Ca biên: OK")

Ca biên: OK


In [13]:
# 12.3 ngẫu nhiên
rng = random.Random(3681)
for _ in range(600):
    a = [rng.randint(0, 31) for _ in range(rng.randint(2, 7))]
    b = brute(a)
    assert all(f(a) == b for f in METHODS[1:]), a
for _ in range(300):
    a = [rng.randint(0, 2**12) for _ in range(rng.randint(2, 40))]
    assert basis_greedy(a) == span_set(a) == gauss_jordan(a)
print("900 test ngẫu nhiên: OK")

900 test ngẫu nhiên: OK


In [14]:
# 12.4 test lớn
rng = random.Random(1)
a = [rng.randint(0, 10**9) for _ in range(10**5)]
t0 = time.time(); x = Solution().maxXorSubsequences(a); t1 = time.time()
print("đáp án:", x, "| thời gian:", round(t1 - t0, 2), "s | khớp Gauss–Jordan:", x == gauss_jordan(a))

đáp án: 1073741823 | thời gian: 0.17 s | khớp Gauss–Jordan: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nghĩ rằng đáp án là `max(nums) ^ max khác` (chỉ ghép hai phần tử)

In [15]:
def wrong_pair(nums):
    return max(a ^ b for a in nums for b in nums + [0])                    # SAI: chỉ tối đa 2 phần tử

print("sai:", wrong_pair([1, 2, 4]), "| đúng:", brute([1, 2, 4]))

sai: 6 | đúng: 7


### Lỗi 2 — Tham lam "XOR nếu bit cao nhất của số chưa bật" trên số gốc (không xây cơ sở)

In [16]:
def wrong_raw_greedy(nums):
    res = 0
    for x in sorted(nums, reverse=True):
        if res ^ x > res: res ^= x                                           # SAI: số gốc không ở dạng bậc thang
    return res

rng = random.Random(7)
while True:
    a = [rng.randint(0, 15) for _ in range(rng.randint(2, 6))]
    if wrong_raw_greedy(a) != brute(a): break
print("phản ví dụ:", a, "| sai:", wrong_raw_greedy(a), "| đúng:", brute(a))

phản ví dụ: [1, 12, 1, 7, 1, 4] | sai: 13 | đúng: 15


### Lỗi 3 — Duyệt bit từ thấp lên khi chèn

In [17]:
def wrong_low_to_high(nums):
    basis = [0] * 30
    for x in nums:
        for b in range(30):                                                  # SAI: phải từ bit cao
            if not x >> b & 1: continue
            if basis[b]: x ^= basis[b]
            else: basis[b] = x; break
    res = 0
    for b in range(29, -1, -1):
        if res ^ basis[b] > res: res ^= basis[b]
    return res

rng = random.Random(8)
while True:
    a = [rng.randint(0, 15) for _ in range(rng.randint(2, 6))]
    if wrong_low_to_high(a) != brute(a): break
print("phản ví dụ:", a, "| sai:", wrong_low_to_high(a), "| đúng:", brute(a))

phản ví dụ: [11, 12, 4] | sai: 8 | đúng: 15


### Lỗi 4 — Thêm ràng buộc không có trong đề (giữ thứ tự, rời nhau, khác rỗng)

Thứ tự không ảnh hưởng XOR, và cho phép chồng nhau làm phần chung triệt tiêu — nên không cần DP theo vị trí. Đề cũng cho phép dãy con **rỗng** (XOR = 0). Nếu tự ép "hai dãy con rời nhau và khác rỗng", với `[7, 7]` ta chỉ còn `7 ⊕ 7 = 0`, trong khi đáp án đúng là `7` (một dãy `[7]`, dãy kia rỗng). Đọc kỹ đề trước khi thêm ràng buộc.

In [18]:
def wrong_disjoint(nums):
    # SAI: ép hai dãy con rời nhau rồi lấy max(X ^ Y) — thực ra vẫn đạt mọi tập con (lấy Y rỗng),
    # nhưng nếu lại ép thêm "cả hai khác rỗng" thì sai với n nhỏ
    n = len(nums); best = 0
    for A in range(1, 1 << n):
        for B in range(1, 1 << n):
            if A & B: continue
            x = reduce(lambda u, v: u ^ v, [nums[i] for i in range(n) if A >> i & 1], 0)
            y = reduce(lambda u, v: u ^ v, [nums[i] for i in range(n) if B >> i & 1], 0)
            best = max(best, x ^ y)
    return best

print("ép rời nhau + khác rỗng:", wrong_disjoint([7, 7]), "| đúng:", brute([7, 7]))

ép rời nhau + khác rỗng: 0 | đúng: 7


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "XOR lớn nhất của một tập con" ⇒ **linear basis** (≤ số bit vectơ).
- Hai lựa chọn XOR với nhau, được chồng nhau ⇒ triệt tiêu ⇒ thường quy về một tập con.
- Câu hỏi khác trên không gian XOR (thứ `k` nhỏ nhất, đếm, kiểm tra thuộc) đều dùng cơ sở rút gọn Gauss–Jordan.
- Cơ sở XOR có thể **gộp** (chèn mọi vectơ của cơ sở này vào cơ sở kia) trong `O(30²)` — hữu ích cho segment tree hoặc DP trên cây lưu cơ sở ở mỗi nút.
- Số bit cần dùng = `max(nums).bit_length()`; với `10^9` là 30.

## 15. Biến thể của bài

1. **XOR tập con nhỏ nhất khác 0**: phần tử cơ sở rút gọn nhỏ nhất.
2. **Giá trị lớn thứ `k`** trong không gian (tính cả 0, các giá trị phân biệt): biểu diễn `k−1` nhị phân trên cơ sở rút gọn.
3. **Số tập con có XOR = `t`**: `2^{n − rank}` nếu `t` thuộc không gian, ngược lại 0.
4. **Truy vấn đoạn** XOR lớn nhất (`nums[l..r]`): cơ sở "tiền tố" lưu vị trí xuất hiện muộn nhất.

In [19]:
# Biến thể 2 + 3
def reduced_basis(nums):
    basis = build_basis(nums)
    for b in range(30):
        if basis[b]:
            for c in range(b + 1, 30):
                if basis[c] >> b & 1: basis[c] ^= basis[b]
    return [v for v in basis if v]                     # tăng dần theo bit trụ

def kth_smallest(nums, k):                              # k = 1 -> 0
    vecs = reduced_basis(nums)
    if k > 2 ** len(vecs): return None
    res = 0; k -= 1
    for i, v in enumerate(vecs):
        if k >> i & 1: res ^= v
    return res

def count_subsets_xor(nums, t):
    basis = build_basis(nums); x = t
    for b in range(29, -1, -1):
        if x >> b & 1 and basis[b]: x ^= basis[b]
    rank = sum(1 for v in basis if v)
    return 2 ** (len(nums) - rank) if x == 0 else 0

rng = random.Random(15)
for _ in range(300):
    a = [rng.randint(0, 31) for _ in range(rng.randint(1, 8))]
    vals = set()
    for mask in range(1 << len(a)):
        x = 0
        for i in range(len(a)):
            if mask >> i & 1: x ^= a[i]
        vals.add(x)
    srt = sorted(vals)
    assert [kth_smallest(a, k) for k in range(1, len(srt) + 1)] == srt
    t = rng.randint(0, 31)
    assert count_subsets_xor(a, t) == sum(1 for mask in range(1 << len(a)) if reduce(lambda u, v: u ^ v, [a[i] for i in range(len(a)) if mask >> i & 1], 0) == t)
print("Biến thể 2, 3 OK")

Biến thể 2, 3 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Mọi cặp dãy con | `O(4^n)` | Kiểm chứng |
| 2. Tập đạt được | DP tập | `O(n 2^rank)` | Giá trị nhỏ |
| 3. Cơ sở + tham lam | Bậc thang | `O(30 n)` | **Lời giải chuẩn** |
| 4. Gauss–Jordan | Rút gọn hoàn toàn | `O(30 n)` | Cần thứ `k`, đếm |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `maxXorSubsequences([8, 8, 8])`.

**Bài 2 (tính tay).** Xây cơ sở cho `[6, 10, 12]` và tìm đáp án.

**Bài 3 (điền TODO).** Viết `rank_of(nums)` = số chiều của không gian XOR.

**Bài 4 (tự viết lại).** Viết `can_make(nums, t)` kiểm tra `t` có là XOR của một tập con không.

**Bài 5 (mini challenge).** Trả lời truy vấn `max XOR` trên đoạn `nums[l..r]` (gợi ý: cơ sở tiền tố, ưu tiên vectơ có vị trí xuất hiện muộn nhất).

In [20]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def rank_of(nums):
    # TODO
    pass

In [21]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def range_max_xor(nums, queries):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Không gian `{0, 8}`. (2) Lớn nhất `8`. (3) `8`.

**Bài 2.** (1) `12 = 1100`, `10 = 1010`, `6 = 0110`. (2) Chèn 6 (bit 2): đặt; 10 (bit 3): đặt; 12 (bit 3): `12^10 = 0110` → bit 2: `^6 = 0` ⇒ phụ thuộc. (3) Tham lam: `10`, rồi `10 ^ 6 = 12` > 10 ⇒ `12`.

**Bài 3.** (1) Xây cơ sở. (2) Đếm vectơ khác 0. (3) Trả về.

**Bài 4.** (1) Khử `t` bằng cơ sở từ bit cao. (2) Kết quả 0 ⇔ thuộc không gian. (3) Kiểm chứng với tập đạt được.

**Bài 5.** (1) Duyệt `r` tăng dần, lưu `basis[b]` và `pos[b]`. (2) Khi chèn, nếu vectơ mới có vị trí muộn hơn thì **đổi chỗ** và tiếp tục khử vectơ cũ. (3) Truy vấn `[l, r]` (xử lý offline theo `r`): chỉ dùng `basis[b]` có `pos[b] ≥ l`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().maxXorSubsequences([8, 8, 8]) == 8
assert Solution().maxXorSubsequences([6, 10, 12]) == 12 == brute([6, 10, 12])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [23]:
# Bài 3, 4
def rank_of_sol(nums):
    return sum(1 for v in build_basis(nums) if v)

def can_make_sol(nums, t):
    return count_subsets_xor(nums, t) > 0

rng = random.Random(4)
for _ in range(300):
    a = [rng.randint(0, 31) for _ in range(rng.randint(1, 8))]
    vals = {0}
    for x in a: vals |= {v ^ x for v in vals}
    assert 2 ** rank_of_sol(a) == len(vals)
    t = rng.randint(0, 31); assert can_make_sol(a, t) == (t in vals)
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [24]:
# Bài 5: max XOR trên đoạn, offline theo r
def range_max_xor_sol(nums, queries):
    order = sorted(range(len(queries)), key=lambda i: queries[i][1])
    basis = [0] * 30; pos = [-1] * 30; ans = [0] * len(queries); qi = 0
    for r, x in enumerate(nums):
        p = r
        for b in range(29, -1, -1):
            if not x >> b & 1: continue
            if not basis[b]: basis[b], pos[b] = x, p; break
            if pos[b] < p:
                basis[b], x = x, basis[b]; pos[b], p = p, pos[b]
            x ^= basis[b]
        while qi < len(order) and queries[order[qi]][1] == r:
            l = queries[order[qi]][0]; res = 0
            for b in range(29, -1, -1):
                if pos[b] >= l and res ^ basis[b] > res: res ^= basis[b]
            ans[order[qi]] = res; qi += 1
    return ans

rng = random.Random(5)
for _ in range(200):
    a = [rng.randint(0, 63) for _ in range(rng.randint(1, 10))]; qs = []
    for _ in range(6):
        l = rng.randrange(len(a)); qs.append((l, rng.randrange(l, len(a))))
    assert range_max_xor_sol(a, qs) == [span_set(a[l:r + 1]) for l, r in qs]
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
basis = [0] * 30
for x in nums:
    for b in 29..0:
        if bit b của x bật:
            if basis[b]: x ^= basis[b]
            else: basis[b] = x; break
res = 0
for b in 29..0: res = max(res, res ^ basis[b])
```

**Checklist:**

- [ ] Đã quy về XOR **một tập con** (hai dãy con triệt tiêu phần chung)?
- [ ] Chèn duyệt bit từ **cao** xuống?
- [ ] Tham lam trên **cơ sở**, không phải số gốc?
- [ ] Đủ 30 bit cho `10^9`?
- [ ] Đã so với vét cạn?
- [ ] Trường hợp mọi số bằng 0 trả về 0?